# Minute-Level Future Direction Baseline (Notebook)

This notebook prepares a baseline pipeline for predicting the next-minute direction of a major futures contract. It follows the provided specification: feature construction, label generation, rolling-window cross-validation, class-weighted linear models, standardized features, and reusable visualization helpers. All comments and printed text are in English to avoid font issues, while figure titles and labels are also in English.

In [ ]:
import os
from pathlib import Path
from typing import List, Tuple, Dict, Optional
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')

# Configure pandas display for easier debugging inside the notebook
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)


## Paths and lightweight configuration

Large data (8-9 GB) lives in the folder `2005年__20250905`. To avoid long runs, set the following knobs:

* `DATA_DIR`: root folder containing CSVs.
* `MAX_FILES`: limit how many CSVs to process (None means all, but keep it small for quick tests).
* `NROWS_PER_FILE`: optional cap on rows per file while prototyping.
* `START_DATE` / `END_DATE`: limit the date interval (inclusive) used after loading.

You can raise these limits later when running the full dataset.

In [ ]:
# Root folder for CSV files
DATA_DIR = Path('2005年__20250905')

# Control how many files to load (set to None to load all files, but that can be very slow)
MAX_FILES: Optional[int] = 1

# Optional cap on rows per file for quick experiments (set to None to load full file)
NROWS_PER_FILE: Optional[int] = 200_000

# Time window filter (inclusive). Use None to skip filtering.
START_DATE: Optional[str] = '2010-01-01'
END_DATE: Optional[str] = '2012-12-31'

# Output folders for cached figures
OUTPUT_DIR = Path('outputs')
FIG_DIR = OUTPUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Figures will be saved under: {FIG_DIR.resolve()}")


## Utility: list CSV files and pick one

We first list available CSVs to choose a subset. Because there may be many files, only the first few names are shown. Adjust `MAX_FILES` above to control how many files will be loaded.

In [ ]:
def list_csv_files(data_dir: Path, limit: Optional[int] = None) -> List[Path]:
    '''Return a sorted list of CSV files under `data_dir`, optionally limited.'''
    files = sorted(data_dir.glob('*.csv'))
    if limit is not None:
        files = files[:limit]
    print(f"Found {len(files)} CSV files (showing up to {limit}):")
    for name in files[:5]:
        print(f"  - {name.name}")
    return files

csv_files = list_csv_files(DATA_DIR, limit=MAX_FILES)
if not csv_files:
    print("No CSV files were found. Please check DATA_DIR.")


## Data loading helper

Loads one CSV at a time with optional row and date filters to keep exploratory runs fast. Progress messages are printed to help monitor long reads.

In [ ]:
def load_single_csv(path: Path, start_date: Optional[str] = None, end_date: Optional[str] = None, nrows: Optional[int] = None) -> pd.DataFrame:
    '''Load a single CSV with optional row and date filtering.'''
    print(f"Loading file: {path.name}")
    df = pd.read_csv(
        path,
        parse_dates=['index'],
        nrows=nrows,
    )
    print(f"Loaded shape before filtering: {df.shape}")

    if start_date is not None:
        df = df[df['index'] >= pd.to_datetime(start_date)]
    if end_date is not None:
        df = df[df['index'] <= pd.to_datetime(end_date)]

    df = df.sort_values('index').reset_index(drop=True)
    print(f"Shape after sorting and date filter: {df.shape}")
    return df


def load_dataset(files: List[Path], start_date: Optional[str], end_date: Optional[str], nrows_per_file: Optional[int]) -> pd.DataFrame:
    '''Concatenate multiple CSV files while applying the same filters.'''
    frames = []
    for idx, path in enumerate(files, start=1):
        print(f"
[Progress] Loading file {idx}/{len(files)}")
        frames.append(load_single_csv(path, start_date=start_date, end_date=end_date, nrows=nrows_per_file))
    if not frames:
        raise ValueError("No data was loaded; please check file list and filters.")
    data = pd.concat(frames, ignore_index=True)
    print(f"Total concatenated shape: {data.shape}")
    return data

if csv_files:
    data_raw = load_dataset(csv_files, START_DATE, END_DATE, NROWS_PER_FILE)
else:
    data_raw = pd.DataFrame()


## Feature engineering

Implements all baseline features described in the specification. Each step adds new columns; missing values from lags or rolling windows are handled later by dropping the earliest rows.

In [ ]:
def compute_return(df: pd.DataFrame) -> pd.Series:
    return np.log(df['close']).diff()


def add_feature_columns(df: pd.DataFrame, lag: int = 5, rsi_window: int = 5, range_window: int = 5) -> pd.DataFrame:
    '''Add baseline features to the provided DataFrame.'''
    df = df.copy()

    # Minute log returns and lags
    df['r_t'] = compute_return(df)
    for k in range(1, lag + 1):
        df[f'r_t_minus_{k}'] = df['r_t'].shift(k)

    # Moving average of close
    df[f'ma_{rsi_window}'] = df['close'].rolling(rsi_window).mean()

    # RSI components
    delta_close = df['close'].diff()
    gain = delta_close.clip(lower=0)
    loss = (-delta_close).clip(lower=0)
    avg_gain = gain.rolling(rsi_window).mean()
    avg_loss = loss.rolling(rsi_window).mean()
    rsi = 100 - 100 / (1 + avg_gain / avg_loss)
    rsi = rsi.fillna(100)  # if avg_loss is zero
    df[f'rsi_{rsi_window}'] = rsi

    # Range features
    df['range_1m'] = df['high'] - df['low']
    rolling_high = df['high'].rolling(range_window).max()
    rolling_low = df['low'].rolling(range_window).min()
    df[f'range_{range_window}'] = rolling_high - rolling_low

    # Volume and open interest changes
    df['volume_change'] = df['volume'].diff()
    df['oi_change'] = df['open_interest'].diff()

    # Spread to VWAP-like avg
    df['spread_vwap'] = df['close'] - df['avg']

    # Additional features
    df['abs_r_t'] = df['r_t'].abs()
    df['r_times_volume'] = df['r_t'] * df['volume']
    df['co_move'] = df['close'] - df['open']

    return df

if not data_raw.empty:
    data_feat = add_feature_columns(data_raw)
else:
    data_feat = pd.DataFrame()

print('Feature columns added. Current columns:')
print(list(data_feat.columns)[:15])


## Label generation

Creates the three-class label with threshold `alpha`. Labels align `X_t` with `Y_{t+1}` by shifting the close price by one minute. Rows with missing features or labels are dropped together.

In [ ]:
def generate_labels(df: pd.DataFrame, alpha: float = 0.001) -> pd.DataFrame:
    '''Generate direction labels using the provided alpha threshold.'''
    df = df.copy()
    close_next = df['close'].shift(-1)
    close_now = df['close']
    ratio = close_next / close_now
    conditions = [ratio > 1 + alpha, ratio < 1 - alpha]
    choices = [1, -1]
    df['label'] = np.select(conditions, choices, default=0)
    return df

if not data_feat.empty:
    data_labeled = generate_labels(data_feat, alpha=0.001)
    # Drop rows with any missing values from feature construction or the shifted label
    data_labeled = data_labeled.dropna().reset_index(drop=True)
else:
    data_labeled = pd.DataFrame()

print(f'Dataset after labeling and dropna: {data_labeled.shape}')
if not data_labeled.empty:
    print(data_labeled[['index', 'close', 'label']].head())


## Feature selection and scaling utilities

Selects the baseline feature set and standardizes using training statistics only. Scaling parameters are reused for the corresponding test fold to avoid leakage.

In [ ]:
BASELINE_FEATURES = [
    'r_t',
    'r_t_minus_1', 'r_t_minus_2', 'r_t_minus_3', 'r_t_minus_4', 'r_t_minus_5',
    'ma_5',
    'rsi_5',
    'range_5',
    'abs_r_t',
    'r_times_volume',
    'co_move',
]


def get_feature_matrix(df: pd.DataFrame, feature_list: List[str]) -> np.ndarray:
    missing = [f for f in feature_list if f not in df.columns]
    if missing:
        raise KeyError(f'Missing features: {missing}')
    return df[feature_list].values


def scale_train_test(X_train: np.ndarray, X_test: np.ndarray) -> Tuple[np.ndarray, np.ndarray, StandardScaler]:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler


## Rolling-window split generator

Splits data in chronological order: 12 months for training and 1 month for testing by default. Adjust `train_months` and `test_months` as needed.

In [ ]:
def month_period(date_series: pd.Series) -> pd.Series:
    return date_series.dt.to_period('M')


def rolling_month_windows(df: pd.DataFrame, train_months: int = 12, test_months: int = 1):
    periods = month_period(df['index'])
    unique_months = periods.unique().sort_values()
    start = 0
    while start + train_months + test_months <= len(unique_months):
        train_month_list = unique_months[start : start + train_months]
        test_month_list = unique_months[start + train_months : start + train_months + test_months]
        train_mask = periods.isin(train_month_list)
        test_mask = periods.isin(test_month_list)
        yield df[train_mask], df[test_mask], train_month_list, test_month_list
        start += test_months


## Model training and evaluation

Uses class-weighted multinomial logistic regression with L2 regularization. Performance is reported for each rolling window and averaged.

In [ ]:
def train_and_evaluate(df: pd.DataFrame, C_values: List[float]) -> pd.DataFrame:
    '''Run rolling-window evaluation across candidate C values.'''
    results = []
    if df.empty:
        print('No data available for training.')
        return pd.DataFrame()

    for C in C_values:
        fold_scores = []
        print(f"
[Info] Evaluating C={C}")
        for fold_idx, (train_df, test_df, train_months, test_months) in enumerate(rolling_month_windows(df), start=1):
            print(f"  Fold {fold_idx}: train {train_months[0]} to {train_months[-1]}, test {test_months[0]}")
            X_train = get_feature_matrix(train_df, BASELINE_FEATURES)
            y_train = train_df['label'].values
            X_test = get_feature_matrix(test_df, BASELINE_FEATURES)
            y_test = test_df['label'].values

            # Scale using training statistics only
            X_train_scaled, X_test_scaled, scaler = scale_train_test(X_train, X_test)

            # Class weights inversely proportional to class frequency
            classes, counts = np.unique(y_train, return_counts=True)
            class_weights = {cls: 1.0 / cnt for cls, cnt in zip(classes, counts)}

            model = LogisticRegression(
                multi_class='multinomial',
                solver='lbfgs',
                max_iter=200,
                C=C,
                class_weight=class_weights,
                n_jobs=-1,
            )
            model.fit(X_train_scaled, y_train)
            preds = model.predict(X_test_scaled)
            acc = accuracy_score(y_test, preds)
            f1 = f1_score(y_test, preds, average='macro')
            fold_scores.append({'fold': fold_idx, 'accuracy': acc, 'f1_macro': f1})
            print(f"    Fold {fold_idx} metrics -> accuracy: {acc:.4f}, macro F1: {f1:.4f}")

        if fold_scores:
            avg_acc = np.mean([fs['accuracy'] for fs in fold_scores])
            avg_f1 = np.mean([fs['f1_macro'] for fs in fold_scores])
            results.append({'C': C, 'avg_accuracy': avg_acc, 'avg_f1_macro': avg_f1})
            print(f"[Summary] C={C} -> avg accuracy: {avg_acc:.4f}, avg macro F1: {avg_f1:.4f}")
    return pd.DataFrame(results)


## Quick sanity run (optional)

Uncomment the following cell to run a light evaluation over a few folds and C values. Keep `MAX_FILES`, `NROWS_PER_FILE`, and the date filters small to avoid long runtimes. This is intended for exploratory checks inside the notebook.

In [ ]:
# Example: run evaluation with small data slices
# if not data_labeled.empty:
#     candidate_C = [0.1, 1.0]
#     cv_results = train_and_evaluate(data_labeled, candidate_C)
#     display(cv_results)


## Visualization helpers

The following helpers create PNG plots (no SVG) and save them under `outputs/figures`. Titles, labels, and legends are in English to avoid font issues. Use them after preparing `data_labeled`.

In [ ]:
def save_and_show(fig, name: str):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f"Saved figure to {path}")


def plot_price_volume(df: pd.DataFrame, title: str = 'Price and Volume Overview'):
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.plot(df['index'], df['close'], color='steelblue', label='Close')
    ax1.set_ylabel('Close Price')
    ax1.set_title(title)
    ax2 = ax1.twinx()
    ax2.bar(df['index'], df['volume'], color='darkorange', alpha=0.3, label='Volume')
    ax2.set_ylabel('Volume')
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')
    save_and_show(fig, 'price_volume_overview')


def plot_class_distribution(df: pd.DataFrame, title: str = 'Label Distribution'):
    counts = df['label'].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(x=counts.index.astype(int), y=counts.values, palette='Blues', ax=ax)
    ax.set_xlabel('Label')
    ax.set_ylabel('Count')
    ax.set_title(title)
    save_and_show(fig, 'label_distribution')


def plot_feature_by_label(df: pd.DataFrame, feature: str, title: Optional[str] = None):
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=df, x='label', y=feature, palette='Set2', ax=ax)
    ax.set_title(title or f'{feature} by label')
    save_and_show(fig, f'feature_{feature}_by_label')


def plot_correlation_heatmap(df: pd.DataFrame, feature_list: List[str]):
    corr = df[feature_list].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(corr, cmap='coolwarm', annot=False, ax=ax)
    ax.set_title('Feature Correlation Heatmap')
    save_and_show(fig, 'feature_correlation_heatmap')

# Example usage (commented to avoid long runtime):
# if not data_labeled.empty:
#     plot_price_volume(data_labeled.head(2000))
#     plot_class_distribution(data_labeled)
#     plot_feature_by_label(data_labeled, 'r_t', 'Return by label')
#     plot_correlation_heatmap(data_labeled, BASELINE_FEATURES)


## Next steps

1. Increase `MAX_FILES`, `NROWS_PER_FILE`, and widen the date range when ready to process more data.
2. Uncomment the evaluation cell to run rolling-window experiments.
3. Generate the recommended visualizations using the helper functions above. All figures are saved as PNG under `outputs/figures`.
4. Consider extending the feature set or testing linear SVM by swapping the model inside `train_and_evaluate`.